# Lesson 3: pandas

**Week 3 · Data Engineering Course**

---

pandas is the most widely used Python library for working with tabular data. It lets you load a CSV file into a **DataFrame** — a table with named columns — and then filter, clean, and summarise it with just a few lines of code.

**Install before starting** (run once in your terminal):
```
pip install pandas
```

**What you will learn:**
- Loading a CSV into a DataFrame
- Exploring the data (shape, types, missing values, unique values)
- Selecting columns and filtering rows
- Cleaning: whitespace, casing, currency symbols, missing values, duplicates
- Grouping and summarising
- Writing the clean result to a new CSV

In [ ]:
import pandas as pd
from pathlib import Path

RAW   = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

print(f'pandas version: {pd.__version__}')

---

## 3.1 Loading a CSV

`pd.read_csv()` reads a CSV file into a DataFrame. It handles headers, types, and encoding automatically.

In [ ]:
df = pd.read_csv(RAW / 'sales_q1.csv')

# Basic information
print(f'Shape: {df.shape}')    # (rows, columns)
print(f'Columns: {list(df.columns)}')

---

## 3.2 Exploring the Data

Before cleaning anything, look at the data. These four steps take less than a minute and tell you almost everything you need to know.

In [ ]:
df.head()   # first 5 rows

In [ ]:
df.info()   # column names, non-null counts, data types

In [ ]:
# How many missing values in each column?
print(df.isnull().sum())

In [ ]:
# What are the unique values in the category column?
# This quickly reveals casing problems and typos
print(df['category'].value_counts())

---

## 3.3 Selecting Columns and Filtering Rows

In [ ]:
# Select one column — returns a Series
print(df['category'].head())

# Select multiple columns — returns a DataFrame
subset = df[['order_id', 'customer_name', 'product', 'total']]
subset.head()

In [ ]:
# Filter rows: keep only Electronics rows
# Step 1: create a True/False mask
mask = df['category'].str.strip().str.upper() == 'ELECTRONICS'

# Step 2: apply it
electronics = df[mask]
print(f'Electronics rows: {len(electronics)}')
electronics[['order_id', 'product', 'category']].head()

In [ ]:
# Combine conditions with & (AND) and | (OR)
# Each condition MUST be in parentheses
high_value = df[
    (df['category'].str.strip().str.upper() == 'ELECTRONICS') &
    (pd.to_numeric(df['total'], errors='coerce') > 200)
]
print(f'High-value Electronics rows: {len(high_value)}')
high_value[['order_id', 'product', 'total']].head()

---

## 3.4 Cleaning Data

Each step below fixes one type of problem. Always work on a copy so the original is preserved.

In [ ]:
# Start with a copy — never modify the original DataFrame
clean = df.copy()

# Step 1: Remove duplicate rows
before = len(clean)
clean = clean.drop_duplicates(subset=['order_id'])
print(f'Duplicates removed: {before - len(clean)}')

In [ ]:
# Step 2: Strip whitespace from text columns
clean['customer_name'] = clean['customer_name'].str.strip()
clean['product']       = clean['product'].str.strip()
print('Whitespace stripped from customer_name and product.')

In [ ]:
# Step 3: Standardise category — strip, title case, fix typos
print('Before:', clean['category'].value_counts().to_dict())

clean['category'] = clean['category'].str.strip().str.title()

print('After:', clean['category'].value_counts().to_dict())

In [ ]:
# Step 4: Clean unit_price — remove $ sign, convert to float
print('unit_price dtype before:', clean['unit_price'].dtype)

clean['unit_price'] = (
    clean['unit_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
clean['unit_price'] = pd.to_numeric(clean['unit_price'], errors='coerce')

print('unit_price dtype after:', clean['unit_price'].dtype)
print('Nulls in unit_price:', clean['unit_price'].isnull().sum())

In [ ]:
# Step 5: Convert quantity to numeric
clean['quantity'] = pd.to_numeric(clean['quantity'], errors='coerce')

# Step 6: Fill in missing totals
clean['total'] = pd.to_numeric(clean['total'], errors='coerce')

missing_before = clean['total'].isnull().sum()
clean['total'] = clean['total'].fillna(clean['quantity'] * clean['unit_price'])
missing_after = clean['total'].isnull().sum()

print(f'Missing totals: {missing_before} before, {missing_after} after')

In [ ]:
# Step 7: Drop rows with invalid quantity
before = len(clean)
clean = clean[clean['quantity'] > 0]
print(f'Rows dropped (quantity <= 0): {before - len(clean)}')
print(f'Clean rows remaining: {len(clean)}')

---

## 3.5 Grouping and Summarising

Groupby works exactly like SQL `GROUP BY`: split the data by a column, apply an aggregation to each group, combine the results.

In [ ]:
# Total revenue and order count by category
revenue = (
    clean
    .groupby('category')['total']
    .agg(orders='count', revenue='sum')
    .sort_values('revenue', ascending=False)
    .round(2)
)
print(revenue)

In [ ]:
# Group by two columns
by_cat_rep = (
    clean
    .groupby(['category', 'sales_rep'])
    .agg(
        orders  = ('order_id', 'count'),
        revenue = ('total', 'sum')
    )
    .round(2)
    .sort_values('revenue', ascending=False)
)
print(by_cat_rep.head(8))

---

## 3.6 Writing to CSV

In [ ]:
# Always use index=False so pandas does not write the row numbers as a column
out_path = CLEAN / 'sales_q1_clean.csv'
clean.to_csv(out_path, index=False)
print(f'Saved {len(clean)} rows to {out_path}')

# Verify by reading back
check = pd.read_csv(out_path)
print(f'Re-read: {check.shape}')
check.head()

---

## Key Takeaways

1. `pd.read_csv()` loads a CSV into a DataFrame. Always check `.shape`, `.info()`, `.isnull().sum()`, and `.value_counts()` before cleaning.
2. Filter rows with a boolean mask in `[]`. Always wrap each condition in parentheses when combining with `&` or `|`.
3. Always call `.copy()` before modifying a DataFrame so the original is preserved.
4. `pd.to_numeric(series, errors='coerce')` converts a column to numbers and turns unparseable values into NaN instead of crashing.
5. `.str.strip()`, `.str.lower()`, `.str.title()` apply to a whole column at once — no loop needed.
6. `.fillna(value)` replaces NaN. Use it to compute missing totals from other columns.
7. `.groupby().agg()` is the pandas equivalent of SQL `GROUP BY`.
8. Always save with `index=False` to avoid a spurious row-number column.